# 第15章　结构化产品与 ABS

[![在 Colab 打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/ch15_structured.ipynb) [![在 Binder 打开](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/ch15_structured.ipynb)

复现例13.1（分层与损失分配）、图15-1、例13.2（现金流瀑布），并对违约率做压力测试。


In [ ]:
# 自举单元：在 Colab/Binder 上自动安装本书复用包 fi；本地运行时自动跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/albertandking/fixed-income.git', '/content/fi-book'], check=False)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/fi-book'], check=False)
    else:
        print('提示：请在仓库根目录执行 `uv sync --extra all` 后再运行本 notebook。')


In [ ]:
import numpy as np
from fi import securitization as sz
from fi import plotting
plotting.use_chinese_style()


## 例13.1　分层、附着/脱离点与损失分配


In [ ]:
tranches = [('优先档', 80), ('夹层档', 15), ('次级档', 5)]
print('附着/脱离点:')
for t in sz.attachment_detachment(tranches):
    print(f"  {t['name']}: 附着={t['attach']*100:.0f}%  脱离={t['detach']*100:.0f}%")
print('\n损失分配（次级先吸收）:')
for loss in (3, 10, 25):
    a = sz.allocate_losses(loss, tranches)
    print(f'  池损失{loss:>2}%: ' + '  '.join(f'{n}={a[n]:.1f}' for n, _ in tranches))


## 图15-1　各档损失率 vs 资产池损失率（编程实验 7）


In [ ]:
sizes = dict(tranches)
pl = np.linspace(0, 30, 121)
series = {n: [] for n, _ in tranches}
for x in pl:
    a = sz.allocate_losses(x, tranches)
    for n, _ in tranches:
        series[n].append(a[n] / sizes[n] * 100)
fig, ax = plotting.new_axes()
for n, _ in tranches:
    ax.plot(pl, series[n], label=n)
ax.set_xlabel('资产池损失率 (%)'); ax.set_ylabel('该档损失率 (%)')
ax.set_title('图15-1　分层与次级垫底'); ax.legend()
fig.tight_layout()


## 例13.2　现金流瀑布：现金不足时次级垫底


In [ ]:
res = sz.sequential_waterfall([12, 12, 4, 70], [('优先档', 80), ('次级档', 20)], [0.04, 0.10])
for n in ['优先档', '次级档']:
    print(f'{n}: 累计利息={sum(res["interest"][n]):.2f}  累计还本={sum(res["principal"][n]):.2f}  期末未偿(损失)={res["shortfall"][n]:.2f}')


## 违约率压力测试（编程实验 8）

资产池违约率从 0% 升到 30%（回收率 40%），看各档损失率如何依次被打穿。


In [ ]:
def stress(default_rate, recovery=0.4):
    pool_loss = 100 * default_rate * (1 - recovery)   # 池损失 = 违约率 × LGD × 池规模
    a = sz.allocate_losses(pool_loss, tranches)
    return {n: a[n] / sizes[n] * 100 for n, _ in tranches}

fig, ax = plotting.new_axes()
drs = np.linspace(0, 0.5, 51)
for n, _ in tranches:
    ax.plot(drs*100, [stress(d)[n] for d in drs], label=n)
ax.set_xlabel('资产池违约率 (%)'); ax.set_ylabel('该档损失率 (%)')
ax.set_title('违约率压力测试（回收率40%）'); ax.legend()
fig.tight_layout()
for d in (0.1, 0.2, 0.35):
    s = stress(d)
    print(f'违约率{d*100:.0f}%: ' + '  '.join(f'{n}={s[n]:.0f}%' for n, _ in tranches))


---

> 小结：分层用次级垫底创造信用——优先档在池损失突破其附着点前零损失；
> 瀑布按优先级分配现金，现金不足时次级先承担损失。分层转移风险，但不消灭风险。
